In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances

print("Semua library berhasil diimpor!")

Semua library berhasil diimpor!


In [5]:
# Memuat data hasil preprocessing
X_scaled = pd.read_csv("../dataset/laptop_features_scaled.csv")

# Memuat hasil clustering FCM
df_fcm = pd.read_csv("../dataset/laptop_fcm_results.csv")

print("Data preprocessing:", X_scaled.shape)
print("Data hasil FCM:", df_fcm.shape)

display(df_fcm.head())

Data preprocessing: (1303, 7)
Data hasil FCM: (1303, 12)


,Price_euros,Ram_GB,Weight_kg,SSD_GB,HDD_GB,CPU_Family_Score,CPU_Base_GHz,Cluster,Membership_Cluster_1,Membership_Cluster_2,Membership_Cluster_3,Membership_Cluster_4
0,0.573361,0.357182,0.169576,0.701024,0.0,0.666667,0.518519,3,0.029546,0.049878,0.880730,0.039846
1,0.461116,0.357182,0.162095,0.000000,0.0,0.666667,0.333333,2,0.215166,0.422223,0.247747,0.114864
2,0.335465,0.357182,0.291771,0.800450,0.0,0.666667,0.592593,3,0.043088,0.071337,0.824008,0.061568
3,0.753120,0.563956,0.284289,0.900155,0.0,1.000000,0.666667,3,0.049493,0.053462,0.783319,0.113726
4,0.657037,0.357182,0.169576,0.800450,0.0,0.666667,0.814815,3,0.051400,0.068501,0.797965,0.082134


In [6]:
# Menggabungkan fitur preprocessing dengan hasil FCM
df_knn = X_scaled.copy()

# Menambahkan label cluster
df_knn["Cluster"] = df_fcm["Cluster"]

# Menambahkan derajat keanggotaan FCM
membership_columns = [
    col for col in df_fcm.columns
    if col.startswith("Membership_")
]

df_knn = pd.concat(
    [df_knn, df_fcm[membership_columns]],
    axis=1
)

print("Penggabungan data berhasil!")
print("Ukuran data KNN:", df_knn.shape)

display(df_knn.head())

Penggabungan data berhasil!
Ukuran data KNN: (1303, 12)


,Price_euros,Ram_GB,Weight_kg,SSD_GB,HDD_GB,CPU_Family_Score,CPU_Base_GHz,Cluster,Membership_Cluster_1,Membership_Cluster_2,Membership_Cluster_3,Membership_Cluster_4
0,0.573361,0.357182,0.169576,0.701024,0.0,0.666667,0.518519,3,0.029546,0.049878,0.880730,0.039846
1,0.461116,0.357182,0.162095,0.000000,0.0,0.666667,0.333333,2,0.215166,0.422223,0.247747,0.114864
2,0.335465,0.357182,0.291771,0.800450,0.0,0.666667,0.592593,3,0.043088,0.071337,0.824008,0.061568
3,0.753120,0.563956,0.284289,0.900155,0.0,1.000000,0.666667,3,0.049493,0.053462,0.783319,0.113726
4,0.657037,0.357182,0.169576,0.800450,0.0,0.666667,0.814815,3,0.051400,0.068501,0.797965,0.082134


In [7]:
# Menentukan fitur yang digunakan oleh KNN
knn_features = [
    "Price_euros",
    "Ram_GB",
    "Weight_kg",
    "SSD_GB",
    "HDD_GB",
    "CPU_Family_Score",
    "CPU_Base_GHz",
    "Membership_Cluster_1",
    "Membership_Cluster_2",
    "Membership_Cluster_3",
    "Membership_Cluster_4"
]

# Membentuk matriks fitur KNN
X_knn = df_knn[knn_features].copy()

print("Fitur KNN berhasil disiapkan!")
print("Jumlah data:", X_knn.shape[0])
print("Jumlah fitur:", X_knn.shape[1])

display(X_knn.head())

Fitur KNN berhasil disiapkan!
Jumlah data: 1303
Jumlah fitur: 11


,Price_euros,Ram_GB,Weight_kg,SSD_GB,HDD_GB,CPU_Family_Score,CPU_Base_GHz,Membership_Cluster_1,Membership_Cluster_2,Membership_Cluster_3,Membership_Cluster_4
0,0.573361,0.357182,0.169576,0.701024,0.0,0.666667,0.518519,0.029546,0.049878,0.880730,0.039846
1,0.461116,0.357182,0.162095,0.000000,0.0,0.666667,0.333333,0.215166,0.422223,0.247747,0.114864
2,0.335465,0.357182,0.291771,0.800450,0.0,0.666667,0.592593,0.043088,0.071337,0.824008,0.061568
3,0.753120,0.563956,0.284289,0.900155,0.0,1.000000,0.666667,0.049493,0.053462,0.783319,0.113726
4,0.657037,0.357182,0.169576,0.800450,0.0,0.666667,0.814815,0.051400,0.068501,0.797965,0.082134


In [8]:
# Menentukan jumlah tetangga terdekat
k = 5

# Inisialisasi model KNN
knn_model = NearestNeighbors(
    n_neighbors=k,
    metric="euclidean",
    algorithm="brute"
)

# Melatih model menggunakan data laptop
knn_model.fit(X_knn)

print("Model KNN berhasil dibentuk!")
print("Jumlah tetangga terdekat (K):", k)
print("Metrik jarak: Euclidean")
print("Jumlah data yang digunakan:", X_knn.shape[0])

Model KNN berhasil dibentuk!
Jumlah tetangga terdekat (K): 5
Metrik jarak: Euclidean
Jumlah data yang digunakan: 1303


In [9]:
# Mengambil salah satu laptop sebagai contoh input
sample_index = 0

# Mengambil spesifikasi laptop sebagai query
query = X_knn.iloc[[sample_index]]

# Mencari 5 laptop terdekat
distances, indices = knn_model.kneighbors(query)

# Menampilkan hasil rekomendasi
print("Hasil pengujian rekomendasi KNN")
print("--------------------------------")

for rank, (idx, distance) in enumerate(
    zip(indices[0], distances[0]), start=1
):
    print(f"Rekomendasi {rank}")
    print("Index laptop:", idx)
    print("Jarak Euclidean:", round(distance, 4))
    print()

Hasil pengujian rekomendasi KNN
--------------------------------
Rekomendasi 1
Index laptop: 0
Jarak Euclidean: 0.0

Rekomendasi 2
Index laptop: 495
Jarak Euclidean: 0.0772

Rekomendasi 3
Index laptop: 989
Jarak Euclidean: 0.1082

Rekomendasi 4
Index laptop: 1072
Jarak Euclidean: 0.1107

Rekomendasi 5
Index laptop: 1087
Jarak Euclidean: 0.1111



In [11]:
# Mengambil 6 tetangga terdekat
distances, indices = knn_model.kneighbors(
    X_knn.iloc[[0]],
    n_neighbors=6
)

# Menghilangkan laptop input dari hasil rekomendasi
recommendation_indices = []
recommendation_distances = []

for idx, distance in zip(indices[0], distances[0]):
    if idx != 0:
        recommendation_indices.append(idx)
        recommendation_distances.append(distance)

    if len(recommendation_indices) == 5:
        break

print("Hasil Rekomendasi Laptop")
print("=" * 50)

for rank, (idx, distance) in enumerate(
    zip(recommendation_indices, recommendation_distances),
    start=1
):
    laptop = df_original.iloc[idx]

    print(f"\nRekomendasi {rank}")
    print("Merek       :", laptop["Company"])
    print("Produk      :", laptop["Product"])
    print("Harga       :", f"€{laptop['Price_euros']}")
    print("RAM         :", laptop["Ram"])
    print("Prosesor    :", laptop["Cpu"])
    print("Penyimpanan :", laptop["Memory"])
    print("GPU         :", laptop["Gpu"])
    print("Jarak       :", round(distance, 4))

Hasil Rekomendasi Laptop

Rekomendasi 1
Merek       : Lenovo
Produk      : Thinkpad X260
Harga       : €1099.0
RAM         : 8GB
Prosesor    : Intel Core i5 6200U 2.3GHz
Penyimpanan : 180GB SSD
GPU         : Intel HD Graphics 520
Jarak       : 0.0772

Rekomendasi 2
Merek       : Dell
Produk      : Latitude 5480
Harga       : €1119.0
RAM         : 8GB
Prosesor    : Intel Core i5 7200U 2.5GHz
Penyimpanan : 128GB SSD
GPU         : Intel HD Graphics 620 
Jarak       : 0.1082

Rekomendasi 3
Merek       : HP
Produk      : EliteBook 820
Harga       : €1199.0
RAM         : 8GB
Prosesor    : Intel Core i5 6200U 2.3GHz
Penyimpanan : 256GB SSD
GPU         : Intel HD Graphics 520
Jarak       : 0.1107

Rekomendasi 4
Merek       : Lenovo
Produk      : ThinkPad T460
Harga       : €1199.0
RAM         : 8GB
Prosesor    : Intel Core i5 6200U 2.3GHz
Penyimpanan : 180GB SSD
GPU         : Intel HD Graphics 520
Jarak       : 0.1111

Rekomendasi 5
Merek       : Apple
Produk      : MacBook Pro
Harga       : €

In [10]:
# Memuat dataset asli
df_original = pd.read_csv(
    "../dataset/laptop_price.csv",
    encoding="latin1"
)

# Menampilkan detail laptop hasil rekomendasi
print("Detail Laptop yang Direkomendasikan")
print("=" * 60)

for rank, (idx, distance) in enumerate(
    zip(indices[0], distances[0]), start=1
):
    laptop = df_original.iloc[idx]

    print(f"\nRekomendasi {rank}")
    print("-" * 40)
    print("Merek       :", laptop["Company"])
    print("Produk      :", laptop["Product"])
    print("Tipe        :", laptop["TypeName"])
    print("Harga       :", f"€{laptop['Price_euros']}")
    print("RAM         :", laptop["Ram"])
    print("Prosesor    :", laptop["Cpu"])
    print("Penyimpanan :", laptop["Memory"])
    print("GPU         :", laptop["Gpu"])
    print("Berat       :", laptop["Weight"])
    print("Jarak       :", round(distance, 4))

Detail Laptop yang Direkomendasikan

Rekomendasi 1
----------------------------------------
Merek       : Apple
Produk      : MacBook Pro
Tipe        : Ultrabook
Harga       : €1339.69
RAM         : 8GB
Prosesor    : Intel Core i5 2.3GHz
Penyimpanan : 128GB SSD
GPU         : Intel Iris Plus Graphics 640
Berat       : 1.37kg
Jarak       : 0.0

Rekomendasi 2
----------------------------------------
Merek       : Lenovo
Produk      : Thinkpad X260
Tipe        : Ultrabook
Harga       : €1099.0
RAM         : 8GB
Prosesor    : Intel Core i5 6200U 2.3GHz
Penyimpanan : 180GB SSD
GPU         : Intel HD Graphics 520
Berat       : 1.3kg
Jarak       : 0.0772

Rekomendasi 3
----------------------------------------
Merek       : Dell
Produk      : Latitude 5480
Tipe        : Notebook
Harga       : €1119.0
RAM         : 8GB
Prosesor    : Intel Core i5 7200U 2.5GHz
Penyimpanan : 128GB SSD
GPU         : Intel HD Graphics 620 
Berat       : 1.6kg
Jarak       : 0.1082

Rekomendasi 4
---------------------